In [1]:
# ============================================================
# PHASE 4B.1 — IDENTIFIER & NESTED-FIELD DIAGNOSTICS
# ============================================================

import os
import json
import ast
import re
import numpy as np
import pandas as pd

from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\Aman\Desktop\TrustGuard\product_scam_ml"
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"

REVIEW_FILE = (
    RAW_DIR
    / "he_associated_review_data"
    / "public_reviews_dataset_cleaned.csv"
)

METADATA_FILE = (
    RAW_DIR
    / "amazon_electronics_metadata_sample.parquet"
)

REPORT_DIR = PROJECT_ROOT / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 4B.1 — IDENTIFIER & NESTED-FIELD DIAGNOSTICS")
print("=" * 80)

print("\nReview dataset:")
print(REVIEW_FILE)

print("\nMetadata dataset:")
print(METADATA_FILE)

PHASE 4B.1 — IDENTIFIER & NESTED-FIELD DIAGNOSTICS

Review dataset:
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\he_associated_review_data\public_reviews_dataset_cleaned.csv

Metadata dataset:
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\data\raw\amazon_electronics_metadata_sample.parquet


In [3]:
# ============================================================
# LOAD DATASETS
# ============================================================

reviews = pd.read_csv(REVIEW_FILE)

metadata = pd.read_parquet(METADATA_FILE)

print("=" * 80)
print("DATASETS LOADED")
print("=" * 80)

print("\nReview dataset shape:")
print(reviews.shape)

print("\nMetadata dataset shape:")
print(metadata.shape)

print("\nReview columns:")
for i, col in enumerate(reviews.columns, 1):
    print(f"{i:2}. {col}")

print("\nMetadata columns:")
for i, col in enumerate(metadata.columns, 1):
    print(f"{i:2}. {col}")

C:\Users\Aman\AppData\Local\Temp\ipykernel_13968\3482840190.py:5: DtypeWarning: Columns (0: fake_review_campaign_start_date) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(REVIEW_FILE)


DATASETS LOADED

Review dataset shape:
(381734, 23)

Metadata dataset shape:
(10000, 16)

Review columns:
 1. asin
 2. review_id
 3. reviewer_id
 4. review_title
 5. review_text
 6. review_rating
 7. review_date
 8. product_title
 9. product_url
10. number_of_helpful
11. number_of_photos
12. photo_thumbnail_urls
13. photo_fullsize_urls
14. asin_url
15. review_url
16. reviewer_url
17. fake_review_campaign_start_date
18. fake_review_product
19. reviewer_classified_fake
20. reviewer_classified_honest
21. reviewer_labeled_fake
22. reviewer_labeled_honest
23. review_is_removed_by_amazon

Metadata columns:
 1. main_category
 2. title
 3. average_rating
 4. rating_number
 5. features
 6. description
 7. price
 8. images
 9. videos
10. store
11. categories
12. details
13. parent_asin
14. bought_together
15. subtitle
16. author


In [4]:
# ============================================================
# IDENTIFIER AUDIT
# ============================================================

print("=" * 80)
print("IDENTIFIER AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# REVIEW IDENTIFIERS
# ------------------------------------------------------------

print("\nREVIEW DATASET")
print("-" * 50)

for col in ["asin", "review_id", "reviewer_id"]:
    if col in reviews.columns:
        print(f"\n{col}")
        print("dtype:", reviews[col].dtype)
        print("missing:", reviews[col].isna().sum())
        print("unique:", reviews[col].nunique(dropna=True))
        print("sample:")
        print(reviews[col].dropna().astype(str).head(10).tolist())


# ------------------------------------------------------------
# METADATA IDENTIFIERS
# ------------------------------------------------------------

print("\n\nMETADATA DATASET")
print("-" * 50)

for col in ["asin", "parent_asin"]:
    if col in metadata.columns:
        print(f"\n{col}")
        print("dtype:", metadata[col].dtype)
        print("missing:", metadata[col].isna().sum())
        print("unique:", metadata[col].nunique(dropna=True))
        print("sample:")
        print(metadata[col].dropna().astype(str).head(10).tolist())
    else:
        print(f"\n{col}: NOT PRESENT")

IDENTIFIER AUDIT

REVIEW DATASET
--------------------------------------------------

asin
dtype: str
missing: 0
unique: 3389
sample:
['B003MROHZW', 'B008DZ6JPO', 'B008DZ6JPO', 'B008DZ6JPO', 'B00GIHF6AS', 'B00GIHF6AS', 'B00GIHF6AS', 'B00PAOFGY6', 'B00PAOFGY6', 'B00PH1MQV8']

review_id
dtype: str
missing: 0
unique: 381734
sample:
['R2RDK8SL1QXN8I', 'R1CSQAGJT0JXO7', 'R1XD30XZ3BXO7F', 'R3L61UVBYHB2QK', 'R36G7QSYO7QRAS', 'R1CM6Y3BKQGPSE', 'R2DETEZ6A2EAEE', 'R2CY6X0D8GB0GY', 'R38HLG99YYM6S6', 'R3LVQ4CXB7F53U']

reviewer_id
dtype: str
missing: 0
unique: 334342
sample:
['AHAHNBDF4GJ26DAL3VS7MGJDNCKA', 'AEZ2DKBYDRPE5O3ENPHV3XRJBMGQ', 'AF52D4YPSCPGLWNYVXZDGG525R2A', 'AFPXQX5HARB76E7GOIUTU32332KQ', 'AFP3AD7L3KWYHHQ4UQAIWJAZX7FQ', 'AH74ZCS65PTXJHOWW46XQUKY46GA', 'AHXQBJ33UXM57LZZ6WWI52S2KZIQ', 'AE6L5PJPPSF5H7SWC3T4U5SS6C2A', 'AEFXBYTFC7GO3QU3KRELTF7BPJ2Q', 'AFJEFSALLQGG3EU3MQTHYE2Y4RRA']


METADATA DATASET
--------------------------------------------------

asin: NOT PRESENT

parent_asin
dtype: s

In [5]:
# ============================================================
# ASIN FORMAT INSPECTION
# ============================================================

def inspect_identifier(series, name, sample_size=20):
    s = series.dropna().astype(str)

    print("=" * 80)
    print(f"IDENTIFIER FORMAT: {name}")
    print("=" * 80)

    print("Count:", len(s))
    print("Unique:", s.nunique())

    lengths = s.str.len()

    print("\nLength distribution:")
    print(lengths.value_counts().sort_index().head(20))

    print("\nMinimum length:", lengths.min())
    print("Maximum length:", lengths.max())

    print("\nExample values:")
    for value in s.drop_duplicates().head(sample_size):
        print(repr(value))


inspect_identifier(
    reviews["asin"],
    "reviews.asin"
)

if "parent_asin" in metadata.columns:
    inspect_identifier(
        metadata["parent_asin"],
        "metadata.parent_asin"
    )

if "asin" in metadata.columns:
    inspect_identifier(
        metadata["asin"],
        "metadata.asin"
    )

IDENTIFIER FORMAT: reviews.asin
Count: 381734
Unique: 3389

Length distribution:
asin
10    381734
Name: count, dtype: int64

Minimum length: 10
Maximum length: 10

Example values:
'B003MROHZW'
'B008DZ6JPO'
'B00GIHF6AS'
'B00PAOFGY6'
'B00PH1MQV8'
'B00SGF5N1M'
'B013CE67W0'
'B019M576XW'
'B01D2UJVGS'
'B01DNMRUIQ'
'B01G4NMIZG'
'B01H3MTVE8'
'B01HCKQG7G'
'B01JKO8724'
'B01LSUQSB0'
'B01LZSA84J'
'B01MTJK06C'
'B01N80S2UX'
'B06WW5ZTZ5'
'B06XBZQ3HX'
IDENTIFIER FORMAT: metadata.parent_asin
Count: 10000
Unique: 10000

Length distribution:
parent_asin
10    10000
Name: count, dtype: int64

Minimum length: 10
Maximum length: 10

Example values:
'B00MCW7G9M'
'B00YT6XQSE'
'B07SM135LS'
'B089CNGZCW'
'B004E2Z88O'
'B00TX536EK'
'B07BJ7ZZL7'
'B005G99O2U'
'B01MCZP7RF'
'B00R6R82HS'
'B000LV7P8I'
'B00CFGZAT8'
'B07PXWYLQX'
'B00ABEQ6TY'
'B07Y6K1PRB'
'B07Y6MRH7G'
'B07848ZT9T'
'B00VBTJCJE'
'B007U1YGOS'
'B0BPLX8B2K'


In [6]:
# ============================================================
# ASIN FORMAT INSPECTION
# ============================================================

def inspect_identifier(series, name, sample_size=20):
    s = series.dropna().astype(str)

    print("=" * 80)
    print(f"IDENTIFIER FORMAT: {name}")
    print("=" * 80)

    print("Count:", len(s))
    print("Unique:", s.nunique())

    lengths = s.str.len()

    print("\nLength distribution:")
    print(lengths.value_counts().sort_index().head(20))

    print("\nMinimum length:", lengths.min())
    print("Maximum length:", lengths.max())

    print("\nExample values:")
    for value in s.drop_duplicates().head(sample_size):
        print(repr(value))


inspect_identifier(
    reviews["asin"],
    "reviews.asin"
)

if "parent_asin" in metadata.columns:
    inspect_identifier(
        metadata["parent_asin"],
        "metadata.parent_asin"
    )

if "asin" in metadata.columns:
    inspect_identifier(
        metadata["asin"],
        "metadata.asin"
    )

IDENTIFIER FORMAT: reviews.asin
Count: 381734
Unique: 3389

Length distribution:
asin
10    381734
Name: count, dtype: int64

Minimum length: 10
Maximum length: 10

Example values:
'B003MROHZW'
'B008DZ6JPO'
'B00GIHF6AS'
'B00PAOFGY6'
'B00PH1MQV8'
'B00SGF5N1M'
'B013CE67W0'
'B019M576XW'
'B01D2UJVGS'
'B01DNMRUIQ'
'B01G4NMIZG'
'B01H3MTVE8'
'B01HCKQG7G'
'B01JKO8724'
'B01LSUQSB0'
'B01LZSA84J'
'B01MTJK06C'
'B01N80S2UX'
'B06WW5ZTZ5'
'B06XBZQ3HX'
IDENTIFIER FORMAT: metadata.parent_asin
Count: 10000
Unique: 10000

Length distribution:
parent_asin
10    10000
Name: count, dtype: int64

Minimum length: 10
Maximum length: 10

Example values:
'B00MCW7G9M'
'B00YT6XQSE'
'B07SM135LS'
'B089CNGZCW'
'B004E2Z88O'
'B00TX536EK'
'B07BJ7ZZL7'
'B005G99O2U'
'B01MCZP7RF'
'B00R6R82HS'
'B000LV7P8I'
'B00CFGZAT8'
'B07PXWYLQX'
'B00ABEQ6TY'
'B07Y6K1PRB'
'B07Y6MRH7G'
'B07848ZT9T'
'B00VBTJCJE'
'B007U1YGOS'
'B0BPLX8B2K'


In [7]:
# ============================================================
# IDENTIFIER NORMALIZATION
# ============================================================

def normalize_asin(value):
    """
    Conservative normalization.

    We ONLY:
    - convert to string
    - strip whitespace
    - uppercase

    We do NOT remove characters or alter the identifier structure.
    """

    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value == "" or value == "NAN":
        return None

    return value


def normalize_series(series):
    return series.map(normalize_asin)


reviews["asin_norm"] = normalize_series(reviews["asin"])

if "parent_asin" in metadata.columns:
    metadata["parent_asin_norm"] = normalize_series(
        metadata["parent_asin"]
    )

if "asin" in metadata.columns:
    metadata["asin_norm"] = normalize_series(
        metadata["asin"]
    )

print("Normalization complete.")

Normalization complete.


In [8]:
# ============================================================
# IDENTIFIER OVERLAP TEST
# ============================================================

review_asins = set(
    reviews["asin"].dropna().astype(str)
)

review_asins_norm = set(
    reviews["asin_norm"].dropna()
)

print("=" * 80)
print("ASIN OVERLAP ANALYSIS")
print("=" * 80)

print("\nUnique review ASINs:")
print(len(review_asins))

print("\nUnique normalized review ASINs:")
print(len(review_asins_norm))


def overlap_report(review_set, metadata_series, name):
    metadata_set = set(
        metadata_series.dropna()
    )

    intersection = review_set.intersection(metadata_set)

    print(f"\n{name}")
    print("-" * 60)
    print("Metadata unique identifiers:", len(metadata_set))
    print("Exact/normalized overlap:", len(intersection))

    if len(review_set) > 0:
        print(
            "Review products matched:",
            f"{len(intersection) / len(review_set) * 100:.4f}%"
        )

    return intersection


matches = {}

if "parent_asin_norm" in metadata.columns:

    matches["review_asin_vs_parent_asin"] = overlap_report(
        review_asins_norm,
        metadata["parent_asin_norm"],
        "review.asin ↔ metadata.parent_asin"
    )


if "asin_norm" in metadata.columns:

    matches["review_asin_vs_metadata_asin"] = overlap_report(
        review_asins_norm,
        metadata["asin_norm"],
        "review.asin ↔ metadata.asin"
    )

ASIN OVERLAP ANALYSIS

Unique review ASINs:
3389

Unique normalized review ASINs:
3389

review.asin ↔ metadata.parent_asin
------------------------------------------------------------
Metadata unique identifiers: 10000
Exact/normalized overlap: 1
Review products matched: 0.0295%


In [28]:
# ============================================================
# UNMATCHED ASIN INSPECTION
# ============================================================

print("=" * 80)
print("UNMATCHED REVIEW ASINS")
print("=" * 80)

if "parent_asin_norm" in metadata.columns:

    metadata_parent_set = set(
        metadata["parent_asin_norm"].dropna()
    )

    unmatched = sorted(
        review_asins_norm - metadata_parent_set
    )

    print(
        "\nUnmatched review products:",
        len(unmatched)
    )

    print("\nFirst 50 unmatched ASINs:")

    for asin in unmatched[:50]:
        print(asin)

UNMATCHED REVIEW ASINS

Unmatched review products: 3388

First 50 unmatched ASINs:
1099134587
1523503033
1698481225
B00003008E
B00004OCJQ
B00004OCNQ
B00004YOBF
B000052XPU
B0000536M2
B000058TJ3
B00005K9CK
B0000631ZM
B00006IC7Q
B00006IV0R
B0000CBJFU
B0000VMEQ2
B0000Z6JJG
B0001EM8I2
B00065XNU8
B0006H92QK
B00076WODS
B0007MHD2Y
B0007NQH98
B0009FHJRS
B000AMA43Q
B000BABW5Q
B000BZA9W8
B000CQBNJY
B000DZF4US
B000F5IKUM
B000F6FPCM
B000FGVQW0
B000FVXSL2
B000FXVAYW
B000GTQU5E
B000J2KEGY
B000JOK11K
B000K6C91M
B000KI111Y
B000KSCEH4
B000L8EEPS
B000LDGNCU
B000LPHVXS
B000MLHMAS
B000MS63E2
B000NPSLNU
B000OLA7KS
B000P0SPX4
B000PEZYNE
B000PUUHIU


In [29]:
# ============================================================
# SEARCH DETAILS FOR ASIN-LIKE IDENTIFIERS
# ============================================================

print("=" * 80)
print("SEARCHING METADATA DETAILS FOR ASIN-LIKE IDENTIFIERS")
print("=" * 80)

asin_pattern = re.compile(
    r"\bB[0-9A-Z]{9}\b",
    re.IGNORECASE
)

def extract_asins_from_value(value):
    if pd.isna(value):
        return []

    text = str(value)

    return list(
        dict.fromkeys(
            m.upper()
            for m in asin_pattern.findall(text)
        )
    )


if "details" in metadata.columns:

    detail_asins = metadata["details"].map(
        extract_asins_from_value
    )

    print("\nMetadata rows containing ASIN-like IDs:")
    print((detail_asins.str.len() > 0).sum())

    print("\nExamples:")

    shown = 0

    for idx, values in detail_asins.items():

        if values:

            print("\nMetadata row:", idx)
            print("Extracted ASINs:", values)
            print("Details:", metadata.loc[idx, "details"])

            shown += 1

            if shown >= 10:
                break

SEARCHING METADATA DETAILS FOR ASIN-LIKE IDENTIFIERS

Metadata rows containing ASIN-like IDs:
110

Examples:

Metadata row: 211
Extracted ASINs: ['BLUELOUNGE']
Details: {"Brand": "Bluelounge", "Item model number": "MA-WH", "Item Weight": "8.1 ounces", "Product Dimensions": "5.24 x 4.25 x 4.76 inches", "Item Dimensions  LxWxH": "5.24 x 4.25 x 4.76 inches", "Color": "White", "Manufacturer": "Bluelounge", "Is Discontinued By Manufacturer": "No", "Date First Available": "December 2, 2015", "Best Sellers Rank": {"Tablet Stands": 4635}, "Compatible Devices": "Apple iPad Mini", "Compatible Phone Models": "Ipad", "Mounting Type": "Tabletop"}

Metadata row: 300
Extracted ASINs: ['BRIEFCASES']
Details: {"Standing screen display size": "15.6 Inches", "Brand": "Lubardy", "Item Weight": "2.18 pounds", "Product Dimensions": "11.81 x 1.57 x 7.87 inches", "Item Dimensions  LxWxH": "11.81 x 1.57 x 7.87 inches", "Color": "Brown", "Department": "Womens", "Manufacturer": "Lubardy", "Country of Origin": "C

In [30]:
# ============================================================
# CANDIDATE IDENTIFIER TABLE
# ============================================================

print("=" * 80)
print("CANDIDATE IDENTIFIER ANALYSIS")
print("=" * 80)

candidate_columns = []

for col in metadata.columns:

    col_lower = col.lower()

    if (
        "asin" in col_lower
        or "product" in col_lower
        or "parent" in col_lower
        or "id" == col_lower
    ):
        candidate_columns.append(col)

print("\nPotential identifier columns:")

for col in candidate_columns:
    print(" -", col)

candidate_results = []

for col in candidate_columns:

    values = normalize_series(metadata[col])

    metadata_set = set(values.dropna())

    overlap = review_asins_norm.intersection(
        metadata_set
    )

    candidate_results.append({
        "metadata_column": col,
        "unique_values": len(metadata_set),
        "review_unique_asins": len(review_asins_norm),
        "matched_review_asins": len(overlap),
        "match_percentage": (
            len(overlap) / len(review_asins_norm) * 100
            if len(review_asins_norm) else 0
        )
    })


candidate_df = pd.DataFrame(candidate_results)

candidate_df = candidate_df.sort_values(
    "match_percentage",
    ascending=False
)

display(candidate_df)

CANDIDATE IDENTIFIER ANALYSIS

Potential identifier columns:
 - parent_asin
 - parent_asin_norm


,metadata_column,unique_values,review_unique_asins,matched_review_asins,match_percentage
0,parent_asin,10000,3389,1,0.029507
1,parent_asin_norm,10000,3389,1,0.029507


In [31]:
# ============================================================
# NESTED FIELD STRUCTURE INSPECTION
# ============================================================

nested_columns = [
    "features",
    "categories",
    "images",
    "videos",
    "details"
]

for col in nested_columns:

    if col not in metadata.columns:
        continue

    print("\n" + "=" * 80)
    print(f"NESTED FIELD: {col}")
    print("=" * 80)

    non_null = metadata[col].dropna()

    print("Non-null:", len(non_null))
    print("Null:", metadata[col].isna().sum())

    print("\nPython types:")

    print(
        non_null.map(type)
        .value_counts()
        .head(10)
    )

    print("\nFirst 5 raw values:")

    for value in non_null.head(5):
        print("\n", repr(value))


NESTED FIELD: features
Non-null: 10000
Null: 0

Python types:
features
<class 'numpy.ndarray'>    10000
Name: count, dtype: int64

First 5 raw values:

 array([], dtype=object)

 array(['UPC: 662774021904', 'Weight: 0.600 lbs'], dtype=object)

 array(['WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13" w/ Touch Bar, 2022/2020 release).',
       'Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macbook from daily scratches, dust, oil, water and fingerprint. Your Macbook remains fresh some years later.',
       'Elegant Style. Our stylish design and printing tech give your Macbook a 360 degree decorative and impressive looking. Take it out and get tons of compliments.',
       'Easy Apply. Easy, bubble-free installation and goo-free removal. Installation guide is well documented in paper material and video format.',
       '100% SATISFACTION GUARANTEED. We use best vinyl material an

In [32]:
# ============================================================
# GENERIC SAFE STRUCTURED-VALUE PARSER
# ============================================================

def parse_structured_value(value):

    if value is None:
        return None

    if isinstance(value, float) and np.isnan(value):
        return None

    # Already structured
    if isinstance(value, (list, dict, tuple)):
        return value

    # String representation of list/dict
    if isinstance(value, str):

        value = value.strip()

        if value == "":
            return None

        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, dict, tuple)):
                return parsed

        except Exception:
            pass

        # JSON fallback
        try:
            parsed = json.loads(value)

            if isinstance(parsed, (list, dict)):
                return parsed

        except Exception:
            pass

        return value

    return value

In [33]:
# ============================================================
# PARSER VALIDATION
# ============================================================

for col in nested_columns:

    if col not in metadata.columns:
        continue

    print("\n" + "=" * 80)
    print(f"PARSER TEST — {col}")
    print("=" * 80)

    for idx, raw_value in metadata[col].dropna().head(5).items():

        parsed = parse_structured_value(raw_value)

        print("\nROW:", idx)

        print("RAW:")
        print(repr(raw_value))

        print("\nPARSED TYPE:")
        print(type(parsed))

        print("\nPARSED:")
        print(repr(parsed))


PARSER TEST — features

ROW: 0
RAW:
array([], dtype=object)

PARSED TYPE:
<class 'numpy.ndarray'>

PARSED:
array([], dtype=object)

ROW: 1
RAW:
array(['UPC: 662774021904', 'Weight: 0.600 lbs'], dtype=object)

PARSED TYPE:
<class 'numpy.ndarray'>

PARSED:
array(['UPC: 662774021904', 'Weight: 0.600 lbs'], dtype=object)

ROW: 2
RAW:
array(['WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13" w/ Touch Bar, 2022/2020 release).',
       'Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macbook from daily scratches, dust, oil, water and fingerprint. Your Macbook remains fresh some years later.',
       'Elegant Style. Our stylish design and printing tech give your Macbook a 360 degree decorative and impressive looking. Take it out and get tons of compliments.',
       'Easy Apply. Easy, bubble-free installation and goo-free removal. Installation guide is well documented in paper material 

In [34]:
# ============================================================
# NESTED FEATURE DERIVATION HELPERS
# ============================================================

def count_list_like(value):

    parsed = parse_structured_value(value)

    if parsed is None:
        return 0

    if isinstance(parsed, (list, tuple)):
        return len(parsed)

    if isinstance(parsed, dict):
        return len(parsed)

    return 0


def text_length_from_structured(value):

    parsed = parse_structured_value(value)

    if parsed is None:
        return 0

    if isinstance(parsed, dict):
        text = " ".join(
            str(v)
            for v in parsed.values()
            if v is not None
        )
        return len(text)

    if isinstance(parsed, (list, tuple)):
        text = " ".join(
            str(v)
            for v in parsed
            if v is not None
        )
        return len(text)

    return len(str(parsed))


def category_count(value):

    parsed = parse_structured_value(value)

    if parsed is None:
        return 0

    if isinstance(parsed, (list, tuple)):
        return len(parsed)

    # Some datasets can contain nested category paths
    if isinstance(parsed, dict):
        return len(parsed)

    # Fallback
    return 1 if str(parsed).strip() else 0


def image_count(value):

    parsed = parse_structured_value(value)

    if parsed is None:
        return 0

    if isinstance(parsed, dict):

        # Prefer hi_res if populated
        for key in ["hi_res", "large", "thumb"]:
            if key in parsed:

                items = parsed[key]

                if isinstance(items, (list, tuple)):
                    return sum(
                        x is not None
                        for x in items
                    )

        return 0

    if isinstance(parsed, (list, tuple)):
        return sum(
            x is not None
            for x in parsed
        )

    return 0


def video_count(value):

    parsed = parse_structured_value(value)

    if parsed is None:
        return 0

    if isinstance(parsed, dict):

        for key in ["url", "title", "user_id"]:

            if key in parsed:

                items = parsed[key]

                if isinstance(items, (list, tuple)):
                    return sum(
                        x is not None
                        for x in items
                    )

        return 0

    if isinstance(parsed, (list, tuple)):
        return sum(
            x is not None
            for x in parsed
        )

    return 0

In [35]:
# ============================================================
# NESTED FEATURE DISTRIBUTION AUDIT
# ============================================================

nested_audit = pd.DataFrame(index=metadata.index)

if "features" in metadata.columns:

    nested_audit["feature_count"] = (
        metadata["features"]
        .map(count_list_like)
    )

    nested_audit["feature_text_length"] = (
        metadata["features"]
        .map(text_length_from_structured)
    )


if "categories" in metadata.columns:

    nested_audit["category_count"] = (
        metadata["categories"]
        .map(category_count)
    )


if "images" in metadata.columns:

    nested_audit["image_count"] = (
        metadata["images"]
        .map(image_count)
    )


if "videos" in metadata.columns:

    nested_audit["video_count"] = (
        metadata["videos"]
        .map(video_count)
    )

    nested_audit["has_videos"] = (
        nested_audit["video_count"] > 0
    ).astype(int)


print("=" * 80)
print("NESTED FEATURE DISTRIBUTIONS")
print("=" * 80)

display(
    nested_audit.describe().T
)

NESTED FEATURE DISTRIBUTIONS


,count,mean,std,min,25%,50%,75%,max
feature_count,10000.0,0.0000,0.00000,0.0,0.0,0.0,0.00,0.0
feature_text_length,10000.0,556.6704,549.02946,2.0,73.0,396.0,901.25,4244.0
category_count,10000.0,1.0000,0.00000,1.0,1.0,1.0,1.00,1.0
image_count,10000.0,0.0000,0.00000,0.0,0.0,0.0,0.00,0.0
video_count,10000.0,0.0000,0.00000,0.0,0.0,0.0,0.00,0.0
has_videos,10000.0,0.0000,0.00000,0.0,0.0,0.0,0.00,0.0


In [36]:
# ============================================================
# CONSTANT FEATURE CHECK
# ============================================================

print("=" * 80)
print("NESTED FEATURE CONSTANT-VALUE CHECK")
print("=" * 80)

for col in nested_audit.columns:

    print(f"\n{col}")

    print("Unique values:",
          nested_audit[col].nunique())

    print("Value counts:")

    print(
        nested_audit[col]
        .value_counts(dropna=False)
        .head(15)
    )

NESTED FEATURE CONSTANT-VALUE CHECK

feature_count
Unique values: 1
Value counts:
feature_count
0    10000
Name: count, dtype: int64

feature_text_length
Unique values: 1913
Value counts:
feature_text_length
2       2202
303       17
304       16
317       16
134       15
130       15
320       15
1045      14
391       14
238       14
262       14
282       14
244       14
375       14
286       14
Name: count, dtype: int64

category_count
Unique values: 1
Value counts:
category_count
1    10000
Name: count, dtype: int64

image_count
Unique values: 1
Value counts:
image_count
0    10000
Name: count, dtype: int64

video_count
Unique values: 1
Value counts:
video_count
0    10000
Name: count, dtype: int64

has_videos
Unique values: 1
Value counts:
has_videos
0    10000
Name: count, dtype: int64


In [37]:
# ============================================================
# PRICE PARSING DIAGNOSTIC
# ============================================================

def parse_price(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    # Remove currency symbols and commas
    cleaned = re.sub(
        r"[^0-9.\-]",
        "",
        text.replace(",", "")
    )

    if cleaned in {"", ".", "-", "-."}:
        return np.nan

    try:
        return float(cleaned)
    except ValueError:
        return np.nan


if "price" in metadata.columns:

    metadata["_price_numeric_test"] = (
        metadata["price"]
        .map(parse_price)
    )

    print("=" * 80)
    print("PRICE PARSING")
    print("=" * 80)

    print("\nRaw price examples:")

    display(
        metadata[
            ["price", "_price_numeric_test"]
        ]
        .drop_duplicates()
        .head(30)
    )

    print("\nPrice availability:")

    print(
        metadata["_price_numeric_test"]
        .notna()
        .value_counts()
    )

    print("\nNumeric price statistics:")

    print(
        metadata["_price_numeric_test"]
        .describe()
    )

PRICE PARSING

Raw price examples:


,price,_price_numeric_test
0,None,NaN
2,19.99,19.99
3,9.99,9.99
4,14.99,14.99
6,14.89,14.89
9,10.99,10.99
14,8.98,8.98
16,909.99,909.99
17,98.49,98.49
19,3.99,3.99



Price availability:
_price_numeric_test
False    5718
True     4282
Name: count, dtype: int64

Numeric price statistics:
count     4282.000000
mean        88.707022
std        335.998426
min          0.930000
25%         11.990000
50%         20.390000
75%         49.990000
max      12998.000000
Name: _price_numeric_test, dtype: float64


In [39]:
# ============================================================
# DIRECT METADATA FEATURE AVAILABILITY — FIXED
# ============================================================

print("=" * 80)
print("DIRECT METADATA FEATURE AVAILABILITY")
print("=" * 80)

metadata_features = [
    "title",
    "description",
    "features",
    "categories",
    "images",
    "videos",
    "store",
    "price",
    "average_rating",
    "rating_number"
]


def safe_unique_count(series):
    """
    Safely calculate unique values even when a column contains
    lists, dictionaries, numpy arrays, etc.
    """

    values = series.dropna()

    # Convert unhashable objects into a stable string representation
    def make_hashable(x):

        if isinstance(x, np.ndarray):
            return repr(x.tolist())

        if isinstance(x, (list, tuple)):
            return repr(x)

        if isinstance(x, dict):
            return repr(sorted(x.items()))

        return x

    try:
        return values.map(make_hashable).nunique(dropna=True)

    except Exception:
        # Final fallback: string representation
        return values.astype(str).nunique(dropna=True)


availability = []

for col in metadata_features:

    if col not in metadata.columns:

        availability.append({
            "column": col,
            "present": False,
            "missing_pct": 100.0,
            "unique_count": 0
        })

        continue

    series = metadata[col]

    availability.append({
        "column": col,
        "present": True,
        "missing_pct": round(
            series.isna().mean() * 100,
            2
        ),
        "unique_count": safe_unique_count(series)
    })


availability_df = pd.DataFrame(availability)

print("\n")
display(availability_df)

DIRECT METADATA FEATURE AVAILABILITY




,column,present,missing_pct,unique_count
0,title,True,0.00,9984
1,description,True,0.00,5597
2,features,True,0.00,7678
3,categories,True,0.00,658
4,images,True,0.00,9915
5,videos,True,0.00,4199
6,store,True,0.62,5887
7,price,True,0.00,1455
8,average_rating,True,0.00,40
9,rating_number,True,0.00,1418


In [40]:
nested_columns = [
    "features",
    "categories",
    "images",
    "videos"
]

print("=" * 80)
print("NESTED FIELD TYPES")
print("=" * 80)

for col in nested_columns:

    print(f"\n--- {col} ---")

    sample = metadata[col].dropna().head(5)

    for value in sample:

        print("Python type:", type(value).__name__)
        print("Value:", repr(value)[:500])
        print()

NESTED FIELD TYPES

--- features ---
Python type: ndarray
Value: array([], dtype=object)

Python type: ndarray
Value: array(['UPC: 662774021904', 'Weight: 0.600 lbs'], dtype=object)

Python type: ndarray
Value: array(['WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13" w/ Touch Bar, 2022/2020 release).',
       'Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macbook from daily scratches, dust, oil, water and fingerprint. Your Macbook remains fresh some years later.',
       'Elegant Style. Our stylish design and printing tech give your Macbook a 360 degree decorative and impressive looking. Take 

Python type: ndarray
Value: array(['☛NotoCity 22mm band is designed for Vivoactive 4 / Samsung Gear S3 Classic / S3 Frontier / Gear Live / Samsung Galaxy Watch 46mm / Huawei Watch GT / Tickwatch Pro / S2 / E2 / Pebble Classic / Time / Time Steel / Pebble 2 (Not Pebble steel) / Other Wat

In [41]:
# ============================================================
# SELECT BEST IDENTIFIER CANDIDATE
# ============================================================

if not candidate_df.empty:

    best_candidate = candidate_df.iloc[0]

    print("=" * 80)
    print("BEST IDENTIFIER CANDIDATE")
    print("=" * 80)

    print(
        "\nColumn:",
        best_candidate["metadata_column"]
    )

    print(
        "Match:",
        f"{best_candidate['match_percentage']:.4f}%"
    )

    if best_candidate["match_percentage"] > 90:

        print(
            "\nSTATUS: STRONG MATCH"
        )

    elif best_candidate["match_percentage"] > 50:

        print(
            "\nSTATUS: PARTIAL MATCH — INVESTIGATE"
        )

    elif best_candidate["match_percentage"] > 0:

        print(
            "\nSTATUS: WEAK MATCH — DO NOT USE YET"
        )

    else:

        print(
            "\nSTATUS: NO MATCH"
        )

BEST IDENTIFIER CANDIDATE

Column: parent_asin
Match: 0.0295%

STATUS: WEAK MATCH — DO NOT USE YET


In [42]:
# ============================================================
# ASIN RELATIONSHIP DIAGNOSTIC
# ============================================================

print("=" * 80)
print("ASIN RELATIONSHIP DIAGNOSTIC")
print("=" * 80)

review_unique = pd.Series(
    list(review_asins_norm),
    name="review_asin"
)

metadata_parent = set(
    metadata["parent_asin_norm"].dropna()
) if "parent_asin_norm" in metadata.columns else set()

metadata_asin = set(
    metadata["asin_norm"].dropna()
) if "asin_norm" in metadata.columns else set()

print("\nReview unique ASINs:", len(review_unique))

print(
    "Metadata parent_asin values:",
    len(metadata_parent)
)

print(
    "Metadata asin values:",
    len(metadata_asin)
)

print(
    "\nReview ASINs in parent_asin:",
    len(review_asins_norm & metadata_parent)
)

print(
    "Review ASINs in metadata asin:",
    len(review_asins_norm & metadata_asin)
)

ASIN RELATIONSHIP DIAGNOSTIC

Review unique ASINs: 3389
Metadata parent_asin values: 10000
Metadata asin values: 0

Review ASINs in parent_asin: 1
Review ASINs in metadata asin: 0


In [43]:
# ============================================================
# SAVE PHASE 4B.1 REPORTS
# ============================================================

candidate_report = (
    REPORT_DIR
    / "phase4b1_identifier_candidates.csv"
)

nested_report = (
    REPORT_DIR
    / "phase4b1_nested_feature_audit.csv"
)

availability_report = (
    REPORT_DIR
    / "phase4b1_metadata_availability.csv"
)

candidate_df.to_csv(
    candidate_report,
    index=False
)

nested_audit.describe().T.to_csv(
    nested_report
)

availability_df.to_csv(
    availability_report,
    index=False
)

print("=" * 80)
print("REPORTS SAVED")
print("=" * 80)

print(candidate_report)
print(nested_report)
print(availability_report)

REPORTS SAVED
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase4b1_identifier_candidates.csv
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase4b1_nested_feature_audit.csv
C:\Users\Aman\Desktop\TrustGuard\product_scam_ml\reports\phase4b1_metadata_availability.csv


In [44]:
# ============================================================
# FINAL PHASE 4B.1 SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PHASE 4B.1 — FINAL DIAGNOSTIC SUMMARY")
print("=" * 80)

review_product_count = reviews["asin_norm"].nunique()

metadata_parent_count = (
    metadata["parent_asin_norm"].nunique()
    if "parent_asin_norm" in metadata.columns
    else 0
)

parent_matches = (
    len(
        review_asins_norm
        & set(metadata["parent_asin_norm"].dropna())
    )
    if "parent_asin_norm" in metadata.columns
    else 0
)

metadata_asin_matches = (
    len(
        review_asins_norm
        & set(metadata["asin_norm"].dropna())
    )
    if "asin_norm" in metadata.columns
    else 0
)

print(f"""
Review products                  : {review_product_count:,}
Metadata rows                    : {len(metadata):,}
Metadata parent ASINs            : {metadata_parent_count:,}

Review → parent_asin matches     : {parent_matches:,}
Review → metadata asin matches   : {metadata_asin_matches:,}

Parent-ASIN match percentage     :
    {parent_matches / review_product_count * 100:.4f}%

Metadata-ASIN match percentage   :
    {metadata_asin_matches / review_product_count * 100:.4f}%
""")

print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

if metadata_asin_matches > parent_matches:

    print("""
metadata.asin currently provides the stronger candidate mapping.
Do NOT automatically replace parent_asin in the production pipeline.
First validate the actual ASIN semantics using matched examples.
""")

elif parent_matches > 0:

    print("""
parent_asin provides some overlap.
Inspect matched and unmatched examples before accepting this mapping.
""")

else:

    print("""
No meaningful direct ASIN mapping was established.

DO NOT continue to model training yet.

The next step is to investigate:
1. Dataset provenance
2. ASIN relationships
3. Possible child/parent product relationships
4. Whether the 10,000 metadata sample and 3,389 review products
   originate from the same Amazon product universe
""")

print("\nNested field unique-value diagnostics:")

for col in nested_audit.columns:

    print(
        f"  {col:25s}: "
        f"{nested_audit[col].nunique():,} unique"
    )

print("\nPhase 4B.1 complete.")


PHASE 4B.1 — FINAL DIAGNOSTIC SUMMARY

Review products                  : 3,389
Metadata rows                    : 10,000
Metadata parent ASINs            : 10,000

Review → parent_asin matches     : 1
Review → metadata asin matches   : 0

Parent-ASIN match percentage     :
    0.0295%

Metadata-ASIN match percentage   :
    0.0000%

INTERPRETATION

parent_asin provides some overlap.
Inspect matched and unmatched examples before accepting this mapping.


Nested field unique-value diagnostics:
  feature_count            : 1 unique
  feature_text_length      : 1,913 unique
  category_count           : 1 unique
  image_count              : 1 unique
  video_count              : 1 unique
  has_videos               : 1 unique

Phase 4B.1 complete.


In [27]:
# ============================================================
# FINAL PHASE 4B.1 SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PHASE 4B.1 — FINAL DIAGNOSTIC SUMMARY")
print("=" * 80)

review_product_count = reviews["asin_norm"].nunique()

metadata_parent_count = (
    metadata["parent_asin_norm"].nunique()
    if "parent_asin_norm" in metadata.columns
    else 0
)

parent_matches = (
    len(
        review_asins_norm
        & set(metadata["parent_asin_norm"].dropna())
    )
    if "parent_asin_norm" in metadata.columns
    else 0
)

metadata_asin_matches = (
    len(
        review_asins_norm
        & set(metadata["asin_norm"].dropna())
    )
    if "asin_norm" in metadata.columns
    else 0
)

print(f"""
Review products                  : {review_product_count:,}
Metadata rows                    : {len(metadata):,}
Metadata parent ASINs            : {metadata_parent_count:,}

Review → parent_asin matches     : {parent_matches:,}
Review → metadata asin matches   : {metadata_asin_matches:,}

Parent-ASIN match percentage     :
    {parent_matches / review_product_count * 100:.4f}%

Metadata-ASIN match percentage   :
    {metadata_asin_matches / review_product_count * 100:.4f}%
""")

print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

if metadata_asin_matches > parent_matches:

    print("""
metadata.asin currently provides the stronger candidate mapping.
Do NOT automatically replace parent_asin in the production pipeline.
First validate the actual ASIN semantics using matched examples.
""")

elif parent_matches > 0:

    print("""
parent_asin provides some overlap.
Inspect matched and unmatched examples before accepting this mapping.
""")

else:

    print("""
No meaningful direct ASIN mapping was established.

DO NOT continue to model training yet.

The next step is to investigate:
1. Dataset provenance
2. ASIN relationships
3. Possible child/parent product relationships
4. Whether the 10,000 metadata sample and 3,389 review products
   originate from the same Amazon product universe
""")

print("\nNested field unique-value diagnostics:")

for col in nested_audit.columns:

    print(
        f"  {col:25s}: "
        f"{nested_audit[col].nunique():,} unique"
    )

print("\nPhase 4B.1 complete.")


PHASE 4B.1 — FINAL DIAGNOSTIC SUMMARY

Review products                  : 3,389
Metadata rows                    : 10,000
Metadata parent ASINs            : 10,000

Review → parent_asin matches     : 1
Review → metadata asin matches   : 0

Parent-ASIN match percentage     :
    0.0295%

Metadata-ASIN match percentage   :
    0.0000%

INTERPRETATION

parent_asin provides some overlap.
Inspect matched and unmatched examples before accepting this mapping.


Nested field unique-value diagnostics:
  feature_count            : 1 unique
  feature_text_length      : 1,913 unique
  category_count           : 1 unique
  image_count              : 1 unique
  video_count              : 1 unique
  has_videos               : 1 unique

Phase 4B.1 complete.


In [45]:
# ============================================================
# PHASE 4B.2 — RAW NESTED OBJECT INSPECTION
# ============================================================

nested_columns = [
    "features",
    "categories",
    "images",
    "videos"
]

print("=" * 80)
print("RAW NESTED OBJECT INSPECTION")
print("=" * 80)

for col in nested_columns:

    print("\n" + "=" * 80)
    print(f"COLUMN: {col}")
    print("=" * 80)

    sample = metadata[col].dropna().head(5)

    for i, value in enumerate(sample):

        print(f"\nExample {i + 1}")
        print("-" * 40)

        print("Python type :", type(value))
        print("repr        :", repr(value)[:1000])

        if hasattr(value, "shape"):
            print("shape       :", value.shape)

        if hasattr(value, "dtype"):
            print("dtype       :", value.dtype)

RAW NESTED OBJECT INSPECTION

COLUMN: features

Example 1
----------------------------------------
Python type : <class 'numpy.ndarray'>
repr        : array([], dtype=object)
shape       : (0,)
dtype       : object

Example 2
----------------------------------------
Python type : <class 'numpy.ndarray'>
repr        : array(['UPC: 662774021904', 'Weight: 0.600 lbs'], dtype=object)
shape       : (2,)
dtype       : object

Example 3
----------------------------------------
Python type : <class 'numpy.ndarray'>
repr        : array(['WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13" w/ Touch Bar, 2022/2020 release).',
       'Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macbook from daily scratches, dust, oil, water and fingerprint. Your Macbook remains fresh some years later.',
       'Elegant Style. Our stylish design and printing tech give your Macbook a 360 degree decorative an

In [46]:
# ============================================================
# DIRECT OBJECT STRUCTURE ANALYSIS
# ============================================================

def inspect_object(value):

    if pd.isna(value) if not isinstance(value, (list, dict, np.ndarray)) else False:
        return {
            "type": "MISSING",
            "length": 0
        }

    if isinstance(value, dict):
        return {
            "type": "dict",
            "length": len(value)
        }

    if isinstance(value, (list, tuple)):
        return {
            "type": type(value).__name__,
            "length": len(value)
        }

    if isinstance(value, np.ndarray):
        return {
            "type": "numpy.ndarray",
            "length": value.size
        }

    if isinstance(value, str):
        return {
            "type": "str",
            "length": len(value)
        }

    return {
        "type": type(value).__name__,
        "length": None
    }


for col in nested_columns:

    print("\n" + "=" * 80)
    print(col)
    print("=" * 80)

    sample = metadata[col].dropna().head(10)

    for value in sample:

        print(
            inspect_object(value),
            "|",
            repr(value)[:300]
        )


features
{'type': 'numpy.ndarray', 'length': 0} | array([], dtype=object)
{'type': 'numpy.ndarray', 'length': 2} | array(['UPC: 662774021904', 'Weight: 0.600 lbs'], dtype=object)
{'type': 'numpy.ndarray', 'length': 5} | array(['WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13" w/ Touch Bar, 2022/2020 release).',
       'Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macbook from daily scratches, dust, oil, w
{'type': 'numpy.ndarray', 'length': 5} | array(['☛NotoCity 22mm band is designed for Vivoactive 4 / Samsung Gear S3 Classic / S3 Frontier / Gear Live / Samsung Galaxy Watch 46mm / Huawei Watch GT / Tickwatch Pro / S2 / E2 / Pebble Classic / Time / Time Steel / Pebble 2 (Not Pebble steel) / Other Watches with 22mm lugs.',
       '➹Wrist Siz
{'type': 'numpy.ndarray', 'length': 3} | array(['New Droid X Essentials Combo Pack',
       'Exclusive Package Incredible Value Worth $1

In [47]:
# ============================================================
# ONE PRODUCT DEEP INSPECTION
# ============================================================

print("=" * 80)
print("ONE PRODUCT DEEP INSPECTION")
print("=" * 80)

row = metadata.iloc[0]

print("\nASIN:")
print(row["parent_asin"])

print("\nTITLE:")
print(row["title"])

for col in nested_columns:

    print("\n" + "-" * 70)
    print(col.upper())
    print("-" * 70)

    value = row[col]

    print("Type:")
    print(type(value))

    print("\nRaw representation:")
    print(repr(value))

ONE PRODUCT DEEP INSPECTION

ASIN:
B00MCW7G9M

TITLE:
FS-1051 FATSHARK TELEPORTER V3 HEADSET

----------------------------------------------------------------------
FEATURES
----------------------------------------------------------------------
Type:
<class 'numpy.ndarray'>

Raw representation:
array([], dtype=object)

----------------------------------------------------------------------
CATEGORIES
----------------------------------------------------------------------
Type:
<class 'numpy.ndarray'>

Raw representation:
array(['Electronics', 'Television & Video', 'Video Glasses'], dtype=object)

----------------------------------------------------------------------
IMAGES
----------------------------------------------------------------------
Type:
<class 'dict'>

Raw representation:
{'hi_res': array([None], dtype=object), 'large': array(['https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg'],
      dtype=object), 'thumb': array(['https://m.media-amazon.com/images/I/41qrX56lsYL._AC_

In [48]:
# ============================================================
# STRING / ARRAY / DICT STRUCTURE CHECK
# ============================================================

for col in nested_columns:

    print("\n" + "=" * 80)
    print(col)
    print("=" * 80)

    type_counts = (
        metadata[col]
        .dropna()
        .map(lambda x: type(x).__name__)
        .value_counts()
    )

    print(type_counts)


features
features
ndarray    10000
Name: count, dtype: int64

categories
categories
ndarray    10000
Name: count, dtype: int64

images
images
dict    10000
Name: count, dtype: int64

videos
videos
dict    10000
Name: count, dtype: int64


In [49]:
# ============================================================
# RAW LENGTH DISTRIBUTIONS
# ============================================================

def raw_length(value):

    if value is None:
        return 0

    if isinstance(value, float) and pd.isna(value):
        return 0

    if isinstance(value, (list, tuple, dict, np.ndarray)):
        return len(value)

    if isinstance(value, str):
        return len(value)

    return 0


raw_distribution = {}

for col in nested_columns:

    raw_distribution[col] = (
        metadata[col]
        .apply(raw_length)
    )


for col, values in raw_distribution.items():

    print("\n" + "=" * 80)
    print(col)
    print("=" * 80)

    print(
        values.describe()
    )

    print(
        "\nUnique lengths:",
        values.nunique()
    )

    print(
        "\nMost common lengths:"
    )

    print(
        values.value_counts()
        .head(10)
    )


features
count    10000.000000
mean         3.735900
std          2.290082
min          0.000000
25%          2.000000
50%          5.000000
75%          5.000000
max         19.000000
Name: features, dtype: float64

Unique lengths: 18

Most common lengths:
features
5     5646
0     2202
4      742
3      368
6      299
1      294
2      132
7      100
8      100
10      52
Name: count, dtype: int64

categories
count    10000.000000
mean         4.107100
std          1.462547
min          0.000000
25%          4.000000
50%          4.000000
75%          5.000000
max          7.000000
Name: categories, dtype: float64

Unique lengths: 7

Most common lengths:
categories
5    3734
4    3186
3    1238
6     808
0     724
2     215
7      95
Name: count, dtype: int64

images
count    10000.0
mean         4.0
std          0.0
min          4.0
25%          4.0
50%          4.0
75%          4.0
max          4.0
Name: images, dtype: float64

Unique lengths: 1

Most common lengths:
images
4    1